In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from pathlib import Path
import matplotlib.pyplot as plt
import requests
import urllib3
from urllib3.exceptions import InsecureRequestWarning
import warnings

urllib3.disable_warnings(InsecureRequestWarning)
warnings.simplefilter(action='ignore', category=pd.errors.ParserWarning)

reeds_path = os.path.expanduser('~/Documents/Github/ReEDS/ReEDS/')

In [2]:
month_list = [
    'January',
    'February',
    'March',
    'April',
    'May',
    'June',
    'July',
    'August',
    'September',
    'October',
    'November',
    'December'
]

In [3]:
# Download state-level degree days
for dd_type in ['cdd', 'hdd']:
    for year in range(1995, 2026):
        for month in month_list:
            if dd_type == 'cdd':
                energy_type = 'cooling'
            else:
                energy_type = 'heating'

            url = f"https://ftp.cpc.ncep.noaa.gov/htdocs/degree_days/weighted/legacy_files/{energy_type}/statesCONUS/{year}/{month}.txt"
            fpath = Path('inputs', f"state_{dd_type}_{month}_{year}.txt")
        
            response = requests.get(url, verify=False)
            if response.status_code == 200:
                with open(fpath, "w", encoding="utf-8") as f:
                    f.write(response.text)
            else:
                print(f"Error: {response.status_code}")

In [6]:
# Get annual HDD/CDDs for each state and each year from 1995 to 2025
cdd_dict = {}
for year in range(1995, 2026):
    cdd_data_for_year = []
    for month in month_list:
        df = (
            pd.read_csv(
                Path('inputs', f'state_cdd_{month}_{year}.txt'),
                skiprows=14,
                sep='\s{2,}| -',
                header=None
            )
            .set_index(0)
            [1]
        )
        cdd_data_for_year.append(df)
    cdd_dict[year] = pd.concat(cdd_data_for_year, axis=1).sum(axis=1)

hdd_dict = {}
for year in range(1995, 2026):
    hdd_data_for_year = []
    for month in month_list:
        df = (
            pd.read_csv(
                Path('inputs', f'state_hdd_{month}_{year}.txt'),
                skiprows=14,
                sep='\s{2,}| -',
                header=None
            )
            .set_index(0)
            [1]
        )
        hdd_data_for_year.append(df)
    hdd_dict[year] = pd.concat(hdd_data_for_year, axis=1).sum(axis=1)

state_cdd = pd.concat(cdd_dict, axis=1).rename_axis(index='state').iloc[:48].transpose()
state_hdd = pd.concat(hdd_dict, axis=1).rename_axis(index='state').iloc[:48].transpose()

In [7]:
# Use 30-year linear trend from 1995 to 2024 to estimate HDD/CDDs for 2026-2050
all_state_cdds = {}
X = np.array(state_cdd.loc[:2024].index).reshape(-1, 1)
for state in state_cdd.columns:
    y = np.array(state_cdd.loc[:2024][state])
    reg = LinearRegression().fit(X, y)
    state_cdd_projected = pd.Series(
        np.arange(2026, 2051) * reg.coef_ + reg.intercept_,
        index = np.arange(2026, 2051)
    )
    all_state_cdds[state] = pd.concat([state_cdd[state], state_cdd_projected])

all_state_hdds = {}
X = np.array(state_hdd.loc[:2024].index).reshape(-1, 1)
for state in state_hdd.columns:
    y = np.array(state_hdd.loc[:2024][state])
    reg = LinearRegression().fit(X, y)
    state_hdd_projected = pd.Series(
        np.arange(2026, 2051) * reg.coef_ + reg.intercept_,
        index = np.arange(2026, 2051)
    )
    all_state_hdds[state] = pd.concat([state_hdd[state], state_hdd_projected])

state_cdd = pd.concat(all_state_cdds, axis=1)
state_cdd.columns = state_cdd.columns.str.title()

state_hdd = pd.concat(all_state_hdds, axis=1)
state_hdd.columns = state_hdd.columns.str.title()

state_cdd = state_cdd.loc[2010:]
state_hdd = state_hdd.loc[2010:]

In [8]:
# Get historical (pre-2025) populations and projections for 2030, 2040, and 2050
# Fill in-between years via linear interpolation
def read_historical_state_populations(fpath, year_range):
    hist_population = pd.read_excel(fpath, skiprows=3)
    hist_population = (
        hist_population.rename(columns={'Unnamed: 0': 'state'})
        .set_index('state')
        [year_range]
        .dropna(subset=year_range[0])
        .iloc[5:]
    )
    hist_population.index = hist_population.index.str.replace('.', '')

    return hist_population

population = pd.read_csv(
    Path('inputs', 'NationalProjections_ProjectedTotalPopulation_2030-2050.csv')
)
population = (
    population.drop(columns='FIPS')
    .rename(columns={'Geography Name': 'state'})
    .set_index('state')
    .drop('United States')
    .replace(',', '', regex=True)
    .astype(int)
)
population.columns = population.columns.astype(int)

hist_populations_2010s = read_historical_state_populations(
    Path('inputs', 'nst-est2020.xlsx'),
    range(2010, 2020)
)
hist_populations_2020s = read_historical_state_populations(
    Path('inputs', 'NST-EST2025-POP.xlsx'),
    range(2020, 2026)
)

population = (
    pd.concat(
        [hist_populations_2010s, hist_populations_2020s, population.drop(columns=2020)],
        axis=1
    )
    .dropna(subset=2050)
    .reindex(columns=range(2010, 2051))
    .interpolate(axis=1)
)

In [9]:
# Calculate annual HDD/CDDs for 2010-2050 for non-cendiv gasregs
# by aggregating state-level HDD/CDDs via population-weighted average
gasreg_state_map = {
    'California': ['California'],
    'Northwest': ['Oregon', 'Washington'],
    'Southwest': ['Arizona', 'New Mexico'],
    'Mountain': ['Montana', 'Idaho', 'Wyoming', 'Nevada', 'Utah', 'Colorado']
}
gasreg_cdds = {}
gasreg_hdds = {}

for gasreg, states in gasreg_state_map.items():
    state_weighted_cdds = []
    state_weighted_hdds = []

    gasreg_population = population.loc[states].sum()
    for state in states:
        weight = population.loc[state] / gasreg_population
        weighted_cdd = state_cdd[state] * weight
        weighted_hdd = state_hdd[state] * weight

        state_weighted_cdds.append(weighted_cdd)
        state_weighted_hdds.append(weighted_hdd)

    gasreg_cdds[gasreg] = pd.concat(state_weighted_cdds, axis=1).sum(axis=1)
    gasreg_hdds[gasreg] = pd.concat(state_weighted_hdds, axis=1).sum(axis=1)

In [10]:
# Read cendiv-level degree days and combine with non-cendiv gasreg degree days
cendiv_number_name_map = {
    '1': 'New England',
    '2': 'Mid-Atlantic',
    '3': 'East North Central',
    '4': 'West North Central',
    '5': 'South Atlantic',
    '6': 'East South Central',
    '7': 'West South Central',
    '8': 'Mountain',
    '9': 'Pacific'
}
annual_degree_days = pd.read_csv(
    Path('inputs', 'kdegday.txt'),
    sep='\s+'
)

annual_degree_days['DD_type'] = 'CDD'
annual_degree_days.iloc[::2, annual_degree_days.columns.get_loc('DD_type')] = 'HDD'

annual_hdd = (
    annual_degree_days.loc[annual_degree_days.DD_type == 'HDD']
    .reset_index(drop=True)
    .drop(columns='DD_type')
    .set_index('Year')
    .loc[2010:]
)
annual_hdd.columns = annual_hdd.columns.map(cendiv_number_name_map)
annual_hdd = annual_hdd.drop(columns=['Pacific', 'Mountain'])
for gasreg, hdd in gasreg_hdds.items():
    annual_hdd[gasreg] = hdd.round().astype(int)

annual_cdd = (
    annual_degree_days.loc[annual_degree_days.DD_type == 'CDD']
    .reset_index(drop=True)
    .drop(columns='DD_type')
    .set_index('Year')
    .loc[2010:]
)
annual_cdd.columns = annual_cdd.columns.map(cendiv_number_name_map)
annual_cdd = annual_cdd.drop(columns=['Pacific', 'Mountain'])
for gasreg, cdd in gasreg_cdds.items():
    annual_cdd[gasreg] = cdd.round().astype(int)

In [11]:
# Reformat and export
df_hdd = annual_hdd.copy()
df_hdd.columns = [col.replace('-', '_').replace(' ', '_') for col in df_hdd.columns]
df_hdd = df_hdd.reset_index().rename(columns={'Year': 't'}).assign(ddtype='HDD')

df_cdd = annual_cdd.copy()
df_cdd.columns = [col.replace('-', '_').replace(' ', '_') for col in df_cdd.columns]
df_cdd = df_cdd.reset_index().rename(columns={'Year': 't'}).assign(ddtype='CDD')

df_out = (
    pd.concat([df_cdd, df_hdd])
    .set_index(['t', 'ddtype'])
    .sort_index()
    .sort_index(axis=1)
)
df_out.to_csv(Path(reeds_path, 'inputs', 'fuelprices', 'gasreg_degree_days.csv'))